# 40 — M3 GPU matrix-free Triad-ATRG pilot

## 判定（2026-07-20 UTC）

**M2 は受理済みです。M3 の探索的 GPU pilot を実行してよいです。**

- accepted M2 run: `M2-20260720T005145Z-dd3e385d0a61`
- M2 exact dense–armillary equality: 64/64
- M2 checkpoint: 14、tensor shard 64、fresh-process resume PASS
- M3 の許可範囲: FP64 GPU matrix-free contraction、fixed-seed RSVD、Triad factorization、profiling
- M3 完了表示: `CORE_REPRODUCED`
- certification status: 常に `NOT_CERTIFIED`

RSVD、特異値減衰、Triad residual、influence proxy は探索値です。これらだけから4D RG、mass gap、熱力学極限、連続極限、`CERTIFIED` を主張しません。


## 永続保存と6時間制限

実行前に永続 volume を確認して設定します。

```bash
export VALIDATED_RG_PERSIST_ROOT=/storage/validated_4d_su2_rg
export VALIDATED_RG_PERSIST_ACK=I_CONFIRM_THIS_PATH_IS_PERSISTENT
# 再開時のみ:
export VALIDATED_RG_M3_RUN_ID=M3-...
```

15分ごと、全phase境界、RSVD/Triadの前後にatomic checkpointを保存します。通常時25%以上、checkpoint時35%以上のGPU空きメモリを要求します。OOM時はsector shardを半分にし、3回失敗で `BLOCKED_RESOURCE` になります。5時間以降は長い処理を開始せず、5時間20分までにGPU stateをCPUへ移して保存し、5時間30分までに返ります。

別セッションでは同じrun IDを設定し、fresh kernelでこのnotebookを上から実行してください。


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'AGENTS.md').is_file():
    raise RuntimeError(f'Project root を特定できません: {PROJECT_ROOT}')
sys.path.insert(0, str(PROJECT_ROOT))

missing = [
    requirement for requirement, module in (
        ('pytest>=8', 'pytest'), ('sympy>=1.12', 'sympy'),
        ('opt_einsum>=3.3', 'opt_einsum'),
    )
    if importlib.util.find_spec(module) is None
]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

import pytest
import torch
print('project:', PROJECT_ROOT)
print('pytest:', pytest.__version__)
print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('実 M3 run には CUDA GPU が必要です。')
print('GPU:', torch.cuda.get_device_properties(0).name)


In [ ]:
import json

from src.m3_config import M3Config
from src.m3_parent import verify_accepted_m2_parent

M3_CONFIG = M3Config()
M2_EVIDENCE = verify_accepted_m2_parent(PROJECT_ROOT, M3_CONFIG)
print(json.dumps({
    '判定': 'M2受理。M3探索実装へ進行可。',
    'parent_run': M3_CONFIG.parent_run_id,
    'parent_hashes': M2_EVIDENCE.hashes,
    'projector_tensors': len(M2_EVIDENCE.projector_tensors),
    'M3_status': 'EXPLORATORY -> CORE_REPRODUCED',
    'certification_status': 'NOT_CERTIFIED',
}, ensure_ascii=False, indent=2))


## M3 pilot の内容

M2の64 projector blockを固定順序で読み、全体729次元のoperatorをglobal matrixとして保持せずsector shardごとに適用します。

1. cuTensorNet/cuQuantumを優先し、未導入時は `opt_einsum + torch CUDA`、次にtorch CUDAへfallback。
2. matrix-free `matvec/rmatvec` をlow-cutoff explicit matrixと照合。
3. adjoint consistencyとpath-cache再利用を検査。
4. fixed seed、FP64、oversampling 16、power iteration 2でRSVD。
5. left/core/rightの三factorへ分解してcheckpoint。
6. influence proxyをscreening用に記録するが、rigorous boundとは扱わない。


In [ ]:
import os
import time

test_started = time.monotonic()
cpu = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', '-m', 'not gpu'],
    cwd=PROJECT_ROOT, check=False,
)
if cpu.returncode != 0:
    raise RuntimeError(f'M0–M3 CPU suite failed: exit={cpu.returncode}')
gpu = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q',
     'tests/test_matrix_free.py', 'tests/test_m3_restart.py', '-m', 'gpu'],
    cwd=PROJECT_ROOT, check=False,
)
if gpu.returncode != 0:
    raise RuntimeError(f'M3 required GPU suite failed: exit={gpu.returncode}')

M3_TEST_REPORT = {
    'accepted_m2_parent': 'PASS',
    'm0_m1_m2_regression_cpu_suite': 'PASS',
    'm3_required_cpu_suite': 'PASS',
    'm3_required_gpu_suite': 'PASS',
    'm3_fresh_process_resume': 'PASS',
    'm3_checkpoint_basis_restore': 'PASS',
    'm3_oom_recovery': 'PASS',
    'elapsed_s': time.monotonic() - test_started,
}
print(json.dumps(M3_TEST_REPORT, indent=2))


In [ ]:
from src.m3_orchestrator import create_or_resume_m3
from src.runtime import environment_info, validate_persistent_root

PERSIST_ROOT = validate_persistent_root()
print(json.dumps(environment_info(), ensure_ascii=False, indent=2))
m3_orchestrator = create_or_resume_m3(
    PERSIST_ROOT, M3_CONFIG, PROJECT_ROOT,
    run_id=os.environ.get('VALIDATED_RG_M3_RUN_ID'),
    test_report=M3_TEST_REPORT,
)
print('再開用: export VALIDATED_RG_M3_RUN_ID=' + m3_orchestrator.state.run_id)
assert m3_orchestrator.state.certification_status == 'NOT_CERTIFIED'
M3_RESULT = m3_orchestrator.run_until_checkpoint()
assert M3_RESULT['certification_status'] == 'NOT_CERTIFIED'
M3_RESULT


In [ ]:
from src.common import read_json

loaded = m3_orchestrator.checkpoints.load_latest(restore_rng=False)
if loaded is None:
    raise RuntimeError('有効なM3 checkpointがありません。')
inspection = {
    'run_id': loaded.state.run_id,
    'phase': loaded.state.phase,
    'checkpoint': str(loaded.path),
    'checkpoint_index': loaded.state.checkpoint_index,
    'done': sum(item.status == 'done' for item in loaded.queue.items.values()),
    'pending': sum(item.status == 'pending' for item in loaded.queue.items.values()),
    'candidate_tensors': sorted(loaded.tensors),
    'skipped_invalid': list(loaded.skipped_invalid),
    'certification_status': loaded.state.certification_status,
}
print(json.dumps(inspection, ensure_ascii=False, indent=2))

report_path = m3_orchestrator.run_root / 'reports/M3_report.json'
if report_path.is_file():
    report = read_json(report_path)
    rsvd = report['results']['M3_RSVD']['result']
    print(json.dumps({
        '判定': 'M3 CORE_REPRODUCED。M4は独立レビュー後に判断。',
        'backend': report['results']['M3_BACKEND_DIAGNOSTIC']['result']['selection'],
        'acceptance_gates': report['acceptance_gates'],
        'singular_values': rsvd['singular_values'],
        'relative_residual': rsvd['relative_residual_frobenius'],
        'influence_proxy': rsvd['influence_proxy'],
        'heuristic_results': report['heuristic_results'],
        'checkpoint': report['checkpoint'],
        'memory': report['memory'],
        'gpu_memory': report['gpu_memory'],
        'certification_status': report['certification_status'],
    }, ensure_ascii=False, indent=2))
else:
    print('M3未完了です。next_session_plan.mdに従って同じrun IDを再開してください。')
